# Hartree–Fock Helyum STO-3G İntegralleri — Markdown LaTeX Formülleri

## 1. Gaussiyen Normalizasyon Sabiti

$$
N_i = \left(\frac{2\alpha_i}{\pi}\right)^{3/4}
$$

---

## 2. Örtüşme İntegrali

$$
S =
\sum_{i,j}
d_i d_j N_i N_j
\left(
\frac{\pi}{\alpha_i+\alpha_j}
\right)^{3/2}
$$

---

## 3. Kinetik Enerji İntegrali

$$
T =
\sum_{i,j}
d_i d_j N_i N_j
\frac{3\alpha_i \alpha_j}{\alpha_i+\alpha_j}
\left(
\frac{\pi}{\alpha_i+\alpha_j}
\right)^{3/2}
$$

---

## 4. Çekirdek Potansiyel Enerjisi

$$
V =
\sum_{i,j}
d_i d_j N_i N_j
\left(
-\frac{2\pi Z}{\alpha_i+\alpha_j}
\right)
$$

---

## 5. Tek Elektron Hamiltonyeni

$$
H_{\text{core}} = T + V
$$

---

## 6. İki Elektron İntegrali (Elektron–Elektron İtimi)

$$
V_{ee} =
\sum_{i,j,k,l}
d_i d_j d_k d_l
N_i N_j N_k N_l
\frac{2\pi^{5/2}}
{
(\alpha_i+\alpha_j)
(\alpha_k+\alpha_l)
\sqrt{\alpha_i+\alpha_j+\alpha_k+\alpha_l}
}
$$

---

## 7. Hartree–Fock Toplam Enerji İfadesi (Kapalı Kabuk He)

$$
E_{\text{HF}} =
2H_{\text{core}} + V_{ee}
$$

---

## 8. Birim Dönüşümü

$$
1\ \text{Hartree} \approx 27.211386\ \text{eV}
$$


# ANALİTİK ÇÖZÜM


In [12]:
import numpy as np
from scipy.optimize import minimize_scalar

def calculate_energy(zeta, Z=2):
    """
    Helyum atomu için varyasyonel enerji hesabı (Hartree biriminde).
    E(zeta) = zeta^2 - 2*Z*zeta + (5/8)*zeta
    """
    # Kinetik Enerji (her elektron için zeta^2/2)
    kinetic = zeta**2
    
    # Çekirdek-Elektron Çekimi (her elektron için -Z*zeta)
    potential_ne = -2 * Z * zeta
    
    # Elektron-Elektron İtmesi (Hartree terimi)
    potential_ee = (5/8) * zeta
    
    return kinetic + potential_ne + potential_ee

# Hartree'den eV'ye dönüşüm sabiti
HARTREE_TO_EV = 27.211386

# Enerjiyi minimize etme
result = minimize_scalar(calculate_energy, args=(2,))
zeta_min = result.x
energy_hartree = result.fun
energy_ev = energy_hartree * HARTREE_TO_EV

print(f"--- Helyum Hartree-Fock (Varyasyonel) ---")
print(f"Efektif Çekirdek Yükü (ζ): {zeta_min:.4f}")
print(f"Minimum Enerji (Hartree): {energy_hartree:.4f} Eh")
print(f"Minimum Enerji (eV):      {energy_ev:.4f} eV")
print("-" * 40)
print(f"Deneysel Değer (yaklaşık): -79.005 eV")

--- Helyum Hartree-Fock (Varyasyonel) ---
Efektif Çekirdek Yükü (ζ): 1.6875
Minimum Enerji (Hartree): -2.8477 Eh
Minimum Enerji (eV):      -77.4887 eV
----------------------------------------
Deneysel Değer (yaklaşık): -79.005 eV


# NÜMERİK ÇÖZÜM

In [13]:
import numpy as np

# 1. Parametreler ve Baz Seti Tanımı (STO-3G for Helium)
# Helyum için zeta = 1.688
alpha = np.array([6.36242139, 1.15892300, 0.31364979]) # Gaussiyen üsleri
d = np.array([0.15432897, 0.53532814, 0.44463454])    # Kontraksiyon katsayıları
N = 3 # 3 Gaussiyen bileşeni

# Normalizasyon sabiti hesaplama
def normalize(a):
    return (2 * a / np.pi)**0.75

norm = normalize(alpha)



# Örtüşme İntegrali (Normalize edilmiş bazlar için 1'dir)
S = 0
for i in range(N):
    for j in range(N):
        S += d[i] * d[j] * norm[i] * norm[j] * (np.pi / (alpha[i] + alpha[j]))**1.5

# Kinetik Enerji T
T = 0
for i in range(N):
    for j in range(N):
        T += d[i] * d[j] * norm[i] * norm[j] * (alpha[i] * alpha[j] / (alpha[i] + alpha[j])) * \
             3 * (np.pi / (alpha[i] + alpha[j]))**1.5

# Çekirdek Potansiyel Enerjisi V (Z=2)
Z = 2
V = 0
for i in range(N):
    for j in range(N):
        V += d[i] * d[j] * norm[i] * norm[j] * (-2 * np.pi * Z / (alpha[i] + alpha[j]))

# Tek Elektron Hamiltoniyeni (H_core)
H_core = T + V

# İki Elektron İntegrali (Elektron-Elektron İtimi - İki elektron aynı orbitalde)
# (11|11) integrali
V_ee = 0
for i in range(N):
    for j in range(N):
        for k in range(N):
            for l in range(N):
                term = (d[i]*d[j]*d[k]*d[l] * norm[i]*norm[j]*norm[k]*norm[l] * 2 * np.pi**2.5 / ((alpha[i]+alpha[j]) * (alpha[k]+alpha[l]) * np.sqrt(alpha[i]+alpha[j]+alpha[k]+alpha[l])))
                V_ee += term

# 3. SCF Döngüsü
# Başlangıç yoğunluk matrisi (P)
P = 0
tol = 1e-8
max_iter = 50
E_old = 0

    
# Birim Dönüşümü
E_ev = E_total * 27.211386
print(f"Hartree Birimi: {E_total:.6f} Eh")
print(f"eV Birimi:      {E_ev:.6f} eV")
print("-" * 50)


Hartree Birimi: -2.807784 Eh
eV Birimi:      -76.403693 eV
--------------------------------------------------
